In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import tifffile

directory = "/mlbio_scratch/anagupta/xenium_preprocessed/donor_tiff_images"

dfs = []
tiffs = {}

files = os.listdir(directory)

for file in files:
    if file.endswith(".ome.tif"):
        filename = file.split('.')[0].removesuffix('_morphology_focus')
        tiff = tifffile.imread(os.path.join(directory, file), is_ome=False, level=0)
        tiffs[filename] = tiff
        print(f"Loaded {directory} with shape {tiff.shape}")

In [ ]:
import polars as pl
data = pl.read_csv('/mlbio_scratch/anagupta/xenium_preprocessed/10xgenomics_alzheimers_disease_mouse_data_shuffled.csv')
print("Loaded data: ", data.shape)
print(data.columns)

# import pandas as pd
# data2 = pd.read_csv('/Users/ananyagupta/MLBio/sliced_data/sliced_data.csv')
# print("Loaded data: ", data.shape)
# print(data.columns)

In [3]:
def window_slicing(tiff, data, window_size=8192, overlap=128):
    h, w = tiff.shape[:2]
    for i in range(0, h, window_size - overlap):
        for j in range(0, w, window_size - overlap):
            window = tiff[i:i+window_size, j:j+window_size]
            if window.shape[0] != window_size or window.shape[1] != window_size:
                window = np.pad(window, ((0, window_size - window.shape[0]), (0, window_size - window.shape[1])), mode='constant', constant_values=0)
            scale_factor = .2125 # um/px
            px_min = j * scale_factor
            px_max = (j + window_size) * scale_factor
            py_min = i * scale_factor
            py_max = (i + window_size) * scale_factor
            # print(px_min, px_max, py_min, py_max)
            # window_data = data[
            #     (data['coord_X'] >= px_min ) &
            #     (data['coord_Y'] >= py_min ) &
            #     (data['coord_X'] < px_max) &
            #     (data['coord_Y'] < py_max)
            # ]
            
            window_data = data.filter(
                (pl.col("coord_X") >= px_min) &
                (pl.col("coord_X") < px_max) &
                (pl.col("coord_Y") >= py_min) &
                (pl.col("coord_Y") < py_max)
            )
            yield window, window_data, px_min, py_min

In [4]:
def normalize(image):
    # Normalize if needed
    if image.dtype != np.uint8:
        min_val = np.min(image)
        max_val = np.max(image)
        if max_val > min_val:
            image_norm = (image - min_val) / (max_val - min_val)
        else:
            image_norm = np.zeros_like(image)
        image_uint8 = (image_norm * 255).astype(np.uint8)
    else:
        image_uint8 = image
    return image_uint8

In [ ]:
sliced_tiffs = {}
sliced_data = {}

donors = data.select('donor').unique()['donor'].to_list()

all_ids = set()
updates = []

for donor in donors:
    donor_data = data.filter(pl.col('donor') == donor)
    tiff = tiffs[donor]
    i = 1
    for window, window_data_df, offset_x, offset_y in window_slicing(tiff, donor_data):
        slice_name = donor + '__' + str(i)
        sliced_tiffs[slice_name] = normalize(window)
        
        if window_data_df.height > 0:
            # Adjust coordinates
            window_data_df = (
                window_data_df
                .with_columns([
                    (pl.col('coord_X') - offset_x).alias('coord_X'),
                    (pl.col('coord_Y') - offset_y).alias('coord_Y'),
                    (pl.col('coord_X') + offset_x).alias('abs_coord_X'),
                    (pl.col('coord_Y') + offset_y).alias('abs_coord_Y'),
                    pl.lit(slice_name).alias('cell_section')
                ])
            )
            
            updates.append(window_data_df.select([
                'cell_id', 'coord_X', 'coord_Y', 'abs_coord_X', 'abs_coord_Y', 'cell_section'
            ]))
        
        sliced_data[slice_name] = window_data_df
        ids = set(window_data_df['cell_id'])
        overlap = all_ids & ids
        # if overlap:
        #     print(f"Overlap found in {donor} {i}")
        #     print(overlap)
        all_ids.update(ids)
        i += 1
    print(f"Sliced {i-1} windows from {tiff.shape}")

print(len(sliced_tiffs))
print(len(sliced_data))

sum = 0

for slice_name, window_data in sliced_data.items():
    sum = sum + len(window_data)
    print(slice_name, len(window_data))

print(sum)

In [ ]:
sliced_data['TgCRND8_2_5__10']

coordx = sliced_data['TgCRND8_2_5__10']['coord_X']
coordy = sliced_data['TgCRND8_2_5__10']['coord_Y']

absx = sliced_data['TgCRND8_2_5__10']['abs_coord_X']
absy = sliced_data['TgCRND8_2_5__10']['abs_coord_Y']

# plot the coordx, coordy
import matplotlib.pyplot as plt
plt.scatter(coordx, coordy)
# set x,y limits from 0
# plt.xlim(0, 8192)
# plt.ylim(0, 8192)
plt.show()

In [ ]:
list(sliced_tiffs.keys()) TgCRND8_2_5__10_0

In [ ]:
image = sliced_tiffs['TgCRND8_2_5__10']

# plot the image
plt.imshow(image, cmap='gray')
plt.show()

In [ ]:
# Join updates_df with a suffix to avoid conflicts
joined = data.join(updates, on='cell_id', how='left', suffix='_upd')

# Replace values only when updates are present, otherwise keep original
updated_df = joined.with_columns([
    pl.coalesce(['coord_X_upd', 'coord_X']).alias('coord_X'),
    pl.coalesce(['coord_Y_upd', 'coord_Y']).alias('coord_Y'),
    pl.col('abs_coord_X_upd').alias('abs_coord_X'),
    pl.col('abs_coord_Y_upd').alias('abs_coord_Y'),
    pl.col('cell_section_upd').alias('cell_section'),
])

# Drop the now-redundant '_upd' columns
final_data = updated_df.drop([c for c in updated_df.columns if c.endswith('_upd')])

In [1]:
import os
import numpy as np

output_dir = "/mlbio_scratch/anagupta/xenium_preprocessed/sliced_data"

In [ ]:
import imageio

for slice_name, window in sliced_tiffs.items():
    if window is None:
        print(slice_name)
        continue
    imageio.imwrite(f"{output_dir}/{slice_name}.png", window)

In [20]:
# save all the sliced tiffs
# for slice_name, window in sliced_tiffs.items():
#     tifffile.imwrite(f"/Users/ananyagupta/MLBio/sliced_tiffs/{slice_name}.tif", window)

import imageio
import io
import tarfile



np.savez_compressed(os.path.join(output_dir, "slice_images.npz"), **{
    name: (window)
    for name, window in sliced_tiffs.items()
})


# tar_file_path = os.path.join(output_dir, f"slice_images.tar")
# with tarfile.open(tar_file_path, "w") as tar:
#     for slice_name, window in sliced_tiffs.items():
#         # imageio.imwrite(f"{output_dir}/{slice_name}.png", window_uint8)
        
#         try:
#             # Save the image to an in-memory buffer
#             img_buffer = io.BytesIO()
#             np.save(img_buffer, window)
#             img_buffer.seek(0)
            
#             # Add the image to the tar file
#             tar_info = tarfile.TarInfo(name=f"{slice_name}.npy")
#             tar_info.size = img_buffer.getbuffer().nbytes
#             tar.addfile(tarinfo=tar_info, fileobj=img_buffer)
#             print(f"Saved {slice_name}")
#         except Exception as e:
#             print(f"Error saving {slice_name}: {e}")
#     print(f"Saved {len(sliced_tiffs)} images")

In [ ]:
# save the concatenated data as csv
data.to_csv(os.path.join(output_dir, 'sliced_data.csv'), index=False)
print(f"Saved {len(data)} rows")

In [2]:
# load images from /mlbio_scratch/anagupta/xenium_preprocessed/sliced_data/slice_images.npz
import numpy as np
import os
images = np.load(os.path.join(output_dir, 'slice_images.npz'), allow_pickle=True)
print("Loaded images: ", images.keys())
# plot the first image
import matplotlib.pyplot as plt
# plt.imshow(images['TgCRND8_5_7__1'])
# plt.show()

Loaded images:  KeysView(<numpy.lib.npyio.NpzFile object at 0x7f6fe4425150>)


In [3]:
import cv2
png_dir = os.path.join(output_dir, 'png_images')    
os.makedirs(png_dir, exist_ok=True)
for slice_name, img in images.items():
    img = (img / img.max() * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(png_dir, f"{slice_name}.png"), img)
    print(f"Saved {slice_name}")


Saved wildtype_5_7__1
Saved wildtype_5_7__2
Saved wildtype_5_7__3
Saved wildtype_5_7__4
Saved wildtype_5_7__5
Saved wildtype_5_7__6
Saved wildtype_5_7__7
Saved wildtype_5_7__8
Saved wildtype_5_7__9
Saved wildtype_5_7__10
Saved wildtype_5_7__11
Saved wildtype_5_7__12
Saved wildtype_5_7__13
Saved wildtype_5_7__14
Saved wildtype_5_7__15
Saved wildtype_5_7__16
Saved wildtype_13_4__1
Saved wildtype_13_4__2
Saved wildtype_13_4__3
Saved wildtype_13_4__4
Saved wildtype_13_4__5
Saved wildtype_13_4__6
Saved wildtype_13_4__7
Saved wildtype_13_4__8
Saved wildtype_13_4__9
Saved wildtype_13_4__10
Saved wildtype_13_4__11
Saved wildtype_13_4__12
Saved wildtype_13_4__13
Saved wildtype_13_4__14
Saved wildtype_13_4__15
Saved wildtype_13_4__16
Saved TgCRND8_17_9__1
Saved TgCRND8_17_9__2
Saved TgCRND8_17_9__3
Saved TgCRND8_17_9__4
Saved TgCRND8_17_9__5
Saved TgCRND8_17_9__6
Saved TgCRND8_17_9__7
Saved TgCRND8_17_9__8
Saved TgCRND8_17_9__9
Saved TgCRND8_17_9__10
Saved TgCRND8_17_9__11
Saved TgCRND8_17_9__12

In [6]:
# save all the images as png
png_dir = os.path.join(output_dir, 'png_images')    
os.makedirs(png_dir, exist_ok=True)
for slice_name, img in images.items():
    plt.imsave(os.path.join(png_dir, f"{slice_name}.png"), img, cmap='gray')
    print(f"Saved {slice_name}")


KeyboardInterrupt: 

In [ ]:
img = images['TgCRND8_5_7__4']

# plot the image
plt.imshow(img, cmap='gray')
plt.show()



In [ ]:
import pandas as pd

sliced_data = pd.read_csv('/mlbio_scratch/anagupta/xenium_preprocessed/sliced_data/sliced_data.csv')
print(sliced_data.head())

In [ ]:
slice_name = 'TgCRND8_5_7__1'

coord_x = sliced_data[sliced_data['cell_section'] == slice_name]['coord_X']
coord_y = sliced_data[sliced_data['cell_section'] == slice_name]['coord_Y']
scale_factor = .2125 # um/px
# construct a 8192x8192 numpy array with the coordinates
coord_array = np.zeros((8192, 8192))
px = coord_y/scale_factor
py = coord_x/scale_factor

# get the int pixels around px, py as size 10*10
px_int = np.floor(px).astype(int)
py_int = np.floor(py).astype(int)

# for x, y in zip(px_int, py_int):
#     px_range = np.arange(x - 5, x + 5)
#     py_range = np.arange(y - 5, y + 5)

#     for i in range(len(px_range)):
#         for j in range(len(py_range)):
#             if px_range[i] >= 0 and py_range[j] >= 0 and px_range[i] < 8192 and py_range[j] < 8192:
#                 coord_array[px_range[i], py_range[j]] = 1
# temp = px_int
# px_int = np.clip(py_int + 500  , 0, 8191)
# py_int = np.clip(temp + 500  , 0, 8191)

coord_array[px_int, py_int] = 1

import matplotlib.pyplot as plt
plt.imshow(coord_array, cmap='gray')
plt.scatter(py_int, px_int, marker='x', color='red', s=10) 
plt.show()

In [ ]:
img = images[slice_name]

import matplotlib.pyplot as plt
plt.imshow(img, cmap='gray')
plt.show()

import cv2
# Gaussian blur
blur = cv2.GaussianBlur(img, (51, 51), sigmaX=0)

plt.imshow(blur, cmap='gray')
plt.show()

In [ ]:
# plot histogram of the img
plt.hist(img.ravel(), bins=256, range=(0, 256))
plt.show()

In [ ]:
import numpy as np
import imageio
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage.feature import peak_local_max
import os

# ---- CONFIG ---- #
image_path = "/path/to/your/8192_image.tif"
image = img
num_samples = 2000  # Adjust to how many synthetic nuclei you want
sigma = 0  # Controls the spatial extent of each cell's "influence"
min_distance = 5  # Minimum distance between real detected peaks
intensity_threshold = 10  # Adjust this to ignore low-intensity noise
# ---------------- #

# Load image
# image = imageio.imread(image_path).astype(np.float32)

# Step 1: Detect bright nuclei-like peaks
coordinates = peak_local_max(
    image,
    min_distance=min_distance,
    threshold_abs=intensity_threshold
)

print(f"Detected {len(coordinates)} bright spots (putative nuclei)")

# Step 2: Create density map (impulses at nuclei locations)
density_map = np.zeros_like(image, dtype=np.float32)
for y, x in coordinates:
    density_map[y, x] = 1.0

# Step 3: Smooth with Gaussian to create spatial probability map
prob_map = gaussian_filter(density_map, sigma=sigma)

# Step 4: Normalize to get a proper probability distribution
prob_map /= prob_map.sum()

# Step 5: Flatten and sample pixel indices based on the probability
flat_probs = prob_map.ravel()
sampled_indices = np.random.choice(
    len(flat_probs), size=num_samples, replace=True, p=flat_probs
)

# Step 6: Convert flat indices to 2D coordinates
H, W = image.shape
ys, xs = np.unravel_index(sampled_indices, (H, W))
coordinates = np.array([[x, y] for x, y in zip(xs, ys)])
point_cloud = np.zeros((8192, 8192))
point_cloud[ys, xs] = 1

# plot the point cloud
# plt.imshow(point_cloud, cmap='gray')
plt.scatter(coordinates[:, 0], coordinates[:, 1], marker='x', color='red', s=5)
plt.show()

# Optional: Add small jitter
# jitter = np.random.normal(scale=0.5, size=point_cloud.shape)
# point_cloud = point_cloud + jitter

# Step 7: Visualize a downsampled crop (8192x8192 is too big to view fully)
# crop = image[2000:3000, 2000:3000]
# crop_points = point_cloud[
#     (point_cloud[:, 0] >= 2000) & (point_cloud[:, 0] < 3000) &
#     (point_cloud[:, 1] >= 2000) & (point_cloud[:, 1] < 3000)
# ]

# plt.figure(figsize=(8, 8))
# plt.imshow(crop, cmap="gray")
# plt.scatter(
#     crop_points[:, 0] - 2000, crop_points[:, 1] - 2000,
#     s=5, color="red", alpha=0.5
# )
# plt.title("Sampled Points over Image Crop")
# plt.axis("off")
# plt.show()

# Optional: Save sampled point cloud
# np.save("sampled_point_cloud.npy", point_cloud)


In [ ]:
# read csv from /mlbio_scratch/anagupta/luna/runs/baseline/2025-06-18_12-05-45/test_results/luna_2025-06-18_12-05-45/model_2025-06-18_epoch_249/TgCRND8_5_7__1_0/metadata_pred.csv

import pandas as pd
dir = '/mlbio_scratch/anagupta/luna/runs/baseline/2025-06-18_12-05-45/test_results/luna_2025-06-18_12-05-45/model_2025-06-18_epoch_249/TgCRND8_5_7__1_0'
output_dir = dir + '/test'
pred_df = pd.read_csv(dir + '/metadata_pred.csv', index_col=0)
true_df = pd.read_csv(dir + '/metadata_true.csv', index_col=0)

print(pred_df.head())
print(true_df.head())

In [ ]:
coordinates2 = pred_df[['coord_X', 'coord_Y']].values

from utils.data.load import (
    cell_class_decoding,
    compute_distance,
    position_normalize,
    to_dataframe,
)

coordinates2 = position_normalize(coordinates2)

print(coordinates2)

In [10]:
import sys
import os
sys.path.append(os.path.abspath('/mlbio_scratch/anagupta/luna/LUNA'))

from metrics.evaluation_plot import plot_scatter_visualization

In [ ]:
unique_classes = true_df['cell_class'].unique()
os.makedirs(output_dir, exist_ok=True)
plot_scatter_visualization(true_df, pred_df, unique_classes, output_dir)

In [ ]:

coordinates3 = coordinates

In [ ]:
from scipy.optimize import linear_sum_assignment
import ot

# Cost matrix
M = ot.dist(coordinates2[:, :2], coordinates[:, :2], metric='euclidean')**2

# One-to-one assignment
row_ind, col_ind = linear_sum_assignment(M)

# Map each predicted point to one target point
matched_pts = []
for row, col in zip(row_ind, col_ind):
    matched_pts.append((pred_df.iloc[row]['cell_class'], coordinates[col, 0], coordinates[col, 1]))
    # create final dataframe with the matched points
    final_df = pd.DataFrame(matched_pts, columns=['cell_class', 'coord_X', 'coord_Y'])
    # final_df.to_csv(output_dir + '/matched_points.csv', index=False)
matched_pts = np.array(matched_pts)

print(matched_pts.shape)
print(final_df.head())

# colors = ['red', 'blue', 'green', 'yellow']
# for i in range(4):
#     plt.scatter(matched_pts[matched_pts[:, 2] == i, 0], matched_pts[matched_pts[:, 2] == i, 1], marker='x', color=colors[i], s=5)
# plt.show()

In [ ]:
unique_classes = true_df['cell_class'].unique()
os.makedirs(output_dir, exist_ok=True)
plot_scatter_visualization(true_df, final_df, unique_classes, output_dir)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

point_cloud_2 = np.zeros((8192, 8192))

# sample randonly 7013 points from the image
coordinates2 = np.random.randint(0, 8192, (7013, 2))
coordinates2 = np.hstack((coordinates2, np.zeros((7013, 1), dtype=int)))  # add z column

print(coordinates2.shape)

for i in range(len(coordinates2)):
    l = 8192
    x, y = coordinates2[i,0], coordinates2[i,1]
    # divide into 2*2 block based on x,y
    if x < l/2 and y < l/2:
        z = 0
    elif x < l/2 and y >= l/2:
        z = 1
    elif x >= l/2 and y < l/2:
        z = 2
    else:
        z = 3
    coordinates2[i, 2] = z
    
point_cloud_2[coordinates[:][1], coordinates[:][0]] = 1

colors = ['red', 'blue', 'green', 'yellow']
for i in range(4):
    plt.scatter(coordinates2[coordinates2[:, 2] == i, 0], coordinates2[coordinates2[:, 2] == i, 1], marker='x', color=colors[i], s=5)
plt.show()


In [ ]:
import ot

a = np.ones((len(coordinates2),)) / len(coordinates2)
b = np.ones((len(coordinates),)) / len(coordinates)

# Compute cost matrix (Euclidean distances)
M = ot.dist(coordinates2[:, :2], coordinates[:, :2], metric='euclidean')**2

# Regularization
reg = 1e-5  # tune this

# Compute Sinkhorn transport plan
T = ot.sinkhorn(a, b, M, reg)

# matched_pts = T @ coordinates
indices = T.argmax(axis=1)  # for each source point, find best match
print(indices.shape)
matched_pts = []
for row in T:
    row = np.clip(row, 1e-12, None)  # avoid underflow
    row /= row.sum()
    idx = np.random.choice(len(coordinates), p=row / row.sum())
    matched_pts.append((coordinates[idx,0], coordinates[idx,1], coordinates2[idx,2]))
matched_pts = np.array(matched_pts)
print(matched_pts.shape)
# remove duplicates
matched_pts = np.unique(matched_pts, axis=0)
print(matched_pts.shape)
print(matched_pts[:10])

colors = ['red', 'blue', 'green', 'yellow']
for i in range(4):
    plt.scatter(matched_pts[matched_pts[:, 2] == i, 0], matched_pts[matched_pts[:, 2] == i, 1], marker='x', color=colors[i], s=5)
plt.show()

In [ ]:
from scipy.optimize import linear_sum_assignment
import ot

# Cost matrix
M = ot.dist(coordinates2[:, :2], coordinates[:, :2], metric='euclidean')**2

# One-to-one assignment
row_ind, col_ind = linear_sum_assignment(M)

# Map each predicted point to one target point
matched_pts = []
for row, col in zip(row_ind, col_ind):
    matched_pts.append((coordinates[col, 0], coordinates[col, 1], coordinates2[row, 2]))
matched_pts = np.array(matched_pts)

print(matched_pts.shape)

colors = ['red', 'blue', 'green', 'yellow']
for i in range(4):
    plt.scatter(matched_pts[matched_pts[:, 2] == i, 0], matched_pts[matched_pts[:, 2] == i, 1], marker='x', color=colors[i], s=5)
plt.show()

In [ ]:
print(matched_pts.shape)

In [ ]:
C1 = ot.dist(coordinates2, coordinates2)
C2 = ot.dist(coordinates, coordinates)
p = np.ones((len(coordinates2),)) / len(coordinates2)
q = np.ones((len(coordinates),)) / len(coordinates)

gw_T = ot.gromov.gromov_wasserstein(C1, C2, p, q, 'square_loss')

print(gw_T.shape)

In [ ]:
row_ind, col_ind = linear_sum_assignment(gw_T)

matched_pts = []
for row, col in zip(row_ind, col_ind):
    matched_pts.append((coordinates[col, 0], coordinates[col, 1], coordinates2[row, 2]))
matched_pts = np.array(matched_pts)

print(matched_pts.shape)

colors = ['red', 'blue', 'green', 'yellow']
for i in range(4):
    plt.scatter(matched_pts[matched_pts[:, 2] == i, 0], matched_pts[matched_pts[:, 2] == i, 1], marker='x', color=colors[i], s=5)
plt.show()